# 🧪 Experiment 001: Baseline Synthetic Data Generation

This experiment establishes baseline performance for synthetic data generation **without** fairness constraints.

## Experiment Overview

| Parameter | Value |
|-----------|-------|
| **Experiment ID** | EXP-001 |
| **Name** | Baseline Generation |
| **Model** | Standard VAE |
| **Dataset** | Synthetic Sample |
| **Fairness** | None |

## Objectives

1. Establish baseline generation quality metrics
2. Document baseline fairness metrics (expected bias)
3. Create reference for comparison with fair methods
4. Validate pipeline functionality

## Setup & Imports

In [ ]:
# Standard imports
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.stats import wasserstein_distance, ks_2samp
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Experiment configuration
EXPERIMENT_ID = "EXP-001"
EXPERIMENT_NAME = "Baseline Generation"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"🧪 Experiment: {EXPERIMENT_ID} - {EXPERIMENT_NAME}")

## 1. Configuration

In [ ]:
CONFIG = {
    'n_samples': 10000,
    'n_features': 7,
    'latent_dim': 32,
    'hidden_dims': [128, 256, 128],
    'epochs': 100,
    'batch_size': 256,
    'learning_rate': 1e-3,
    'n_synthetic': 5000
}

print("📋 Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 2. Data Preparation

We generate a synthetic dataset with **intentional gender bias** in the income outcome.
This serves as the baseline dataset that our fair methods will later aim to de-bias.

In [ ]:
# Generate synthetic dataset with intentional bias
n = CONFIG['n_samples']

df = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'gender': np.random.choice(['Male', 'Female'], n, p=[0.55, 0.45]),
    'education_num': np.random.randint(6, 20, n),
    'hours_per_week': np.random.normal(40, 10, n).clip(1, 99),
    'capital_gain': np.random.exponential(500, n).clip(0, 50000),
    'capital_loss': np.random.exponential(100, n).clip(0, 5000),
    'work_experience': np.random.exponential(10, n).clip(0, 50),
})

# Add biased outcome
df['income_prob'] = 0.25 + (df['age'] - 40) / 100 + (df['education_num'] - 12) / 50
df.loc[df['gender'] == 'Female', 'income_prob'] -= 0.15
df['income_prob'] = df['income_prob'].clip(0.05, 0.95)
df['income'] = (np.random.random(n) < df['income_prob']).astype(int)
df = df.drop('income_prob', axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nOutcome by gender:")
print(df.groupby('gender')['income'].mean())

## 3. Model Definition

Standard VAE architecture with no fairness constraints — used as the baseline generative model.

In [ ]:
class BaselineVAE(nn.Module):
    """Standard VAE without fairness constraints."""
    
    def __init__(self, input_dim, latent_dim, hidden_dims):
        super().__init__()
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.LeakyReLU(0.2),
            ])
            prev_dim = h_dim
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_mu = nn.Linear(prev_dim, latent_dim)
        self.fc_var = nn.Linear(prev_dim, latent_dim)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for h_dim in hidden_dims[::-1]:
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.LeakyReLU(0.2),
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_var(h)
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var
    
    def sample(self, n, device):
        z = torch.randn(n, self.fc_mu.out_features).to(device)
        return self.decode(z)

print("✅ Model defined")

## 4. Training

We train the VAE using a standard reconstruction + KL-divergence loss.

In [ ]:
# Prepare data
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])

feature_cols = ['age', 'gender_encoded', 'education_num', 'hours_per_week', 
                'capital_gain', 'capital_loss', 'work_experience']

scaler = StandardScaler()
X = scaler.fit_transform(df[feature_cols])

X_tensor = torch.FloatTensor(X)
dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

# Create model
model = BaselineVAE(X.shape[1], CONFIG['latent_dim'], CONFIG['hidden_dims']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
history = {'loss': [], 'recon': [], 'kl': []}

for epoch in range(CONFIG['epochs']):
    model.train()
    epoch_loss = 0
    
    for (batch_x,) in dataloader:
        batch_x = batch_x.to(device)
        x_recon, mu, log_var = model(batch_x)
        
        recon = F.mse_loss(x_recon, batch_x, reduction='sum')
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        loss = recon + kl
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    history['loss'].append(epoch_loss / len(dataset))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{CONFIG['epochs']} - Loss: {history['loss'][-1]:.4f}")

print("\n✅ Training complete!")

## 5. Evaluation

Generate synthetic data and compute fidelity metrics (Wasserstein distance, KS statistic).

In [ ]:
# Generate synthetic data
model.eval()
with torch.no_grad():
    synthetic = model.sample(CONFIG['n_synthetic'], device).cpu().numpy()

synth_df = pd.DataFrame(scaler.inverse_transform(synthetic), columns=feature_cols)
synth_df['gender'] = np.where(synth_df['gender_encoded'] > 0.5, 'Male', 'Female')

print(f"Synthetic shape: {synth_df.shape}")

In [ ]:
# Fidelity metrics
fidelity_scores = {}
for col in feature_cols:
    wd = wasserstein_distance(df[col].values, synth_df[col].values)
    ks, _ = ks_2samp(df[col].values, synth_df[col].values)
    fidelity_scores[col] = {'wasserstein': wd, 'ks': ks}

avg_wd = np.mean([s['wasserstein'] for s in fidelity_scores.values()])
avg_ks = np.mean([s['ks'] for s in fidelity_scores.values()])
fidelity_score = 1 - (avg_wd + avg_ks) / 2

print(f"Fidelity Score: {fidelity_score:.4f}")
print(f"Avg Wasserstein: {avg_wd:.4f}")
print(f"Avg KS Stat: {avg_ks:.4f}")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, col in enumerate(feature_cols[:8]):
    ax = axes.flatten()[idx]
    ax.hist(df[col], bins=30, alpha=0.5, label='Real', density=True)
    ax.hist(synth_df[col], bins=30, alpha=0.5, label='Synthetic', density=True)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Real vs Synthetic (Baseline)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Results Summary

In [ ]:
results = {
    'experiment_id': EXPERIMENT_ID,
    'experiment_name': EXPERIMENT_NAME,
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'metrics': {
        'fidelity_score': fidelity_score,
        'avg_wasserstein': avg_wd,
        'avg_ks_stat': avg_ks,
        'final_loss': history['loss'][-1]
    },
    'observations': [
        "Baseline VAE learns distribution effectively",
        "No fairness constraints - bias preserved",
        "Reference for fair methods comparison"
    ]
}

print("=" * 60)
print("EXPERIMENT 001 SUMMARY: BASELINE")
print("=" * 60)
print(f"Fidelity: {fidelity_score:.4f}")
print(f"Bias Status: PRESERVED (no fairness constraints)")
print("\nNext: Run EXP-002 with fairness constraints")